# Session 2: Cypher — Da Basi a Query Reali

Benvenuti alla Sessione 2! Qui approfondiremo Cypher sul dataset Org Graph caricato in Sessione 1.

**Cosa faremo:**
1. Connessione a Neo4j e ricarica dataset
2. Pattern MATCH avanzati (path variables, multi-hop)
3. MERGE vs CREATE: idempotenza
4. Filtri WHERE (AND, OR, IN, CONTAINS)
5. Aggregazioni e subquery
6. Date, liste, map
7. Query tuning con PROFILE / EXPLAIN
8. APOC essentials
9. Esercizi

**Dataset:** Org Graph (238 persone, 93 progetti, 55 skill)


## 1. Setup e Connessione


Installiamo le dipendenze e connettiamoci al database Neo4j.
Il driver v6+ supporta `execute_query()` che useremo in tutto il notebook.


In [ ]:
# Installa le dipendenze necessarie
%pip install neo4j python-dotenv pandas -q


In [ ]:
# Import librerie
from neo4j import GraphDatabase
import pandas as pd
import os
from dotenv import load_dotenv

# Carica credenziali dal file .env (root del progetto)
load_dotenv("../../.env")

# Legge URI, username, password
URI  = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PASS = os.getenv("NEO4J_PASSWORD")

# Crea il driver (pool di connessioni thread-safe)
driver = GraphDatabase.driver(URI, auth=(USER, PASS))

# Verifica la connessione
driver.verify_connectivity()
print("Connesso a Neo4j!")


## 2. Recap: il Dataset Org Graph


Se non avete ancora caricato il dataset in Sessione 1, eseguite le celle qui sotto.
Se è già presente, MERGE non creerà duplicati.
Esploriamo il CSV per capire la struttura.


In [ ]:
# Legge il file CSV
df = pd.read_csv("../../data/org_graph.csv")
print(f"Righe: {len(df)}, Colonne: {len(list(df.columns))}")
print(f"Colonne:\n{list(df.columns)}")


In [ ]:
# Le colonne "core" descrivono persona, progetto e supervisor
core_cols = [
    "employee_id","name","gender","discipline","department",
    "code","location","title","supervisor_id","supervisor_name",
    "project_id","project_name","role"
]
# Le restanti colonne sono skill con livello (Novice/.../Master)
skill_cols = [c for c in df.columns if c not in core_cols]
print(f"Core: {len(core_cols)}, Skill: {len(skill_cols)}")


## 3. Pattern MATCH Avanzati


**Path a lunghezza variabile** con `[*min..max]` permette di navigare la profondità del grafo:
- `-[:REPORTS_TO*1..4]->` : da 1 a 4 livelli di gerarchia
- `-[:WORKED_ON*]-(b)` : profondità illimitata

**shortestPath()** : cammino minimo tra due nodi.

**Path variables** : l'intero percorso assegnato a una variabile.


In [ ]:
# --- Catena di comando (path a lunghezza variabile) ---
# REPORTS_TO*1..4 = da 1 a 4 salti nella gerarchia aziendale
# length(path) = numero di relazioni percorse
query = """
MATCH path = (p:Person)-[:REPORTS_TO*1..4]->(m:Person)
RETURN p.name AS impiegato, m.name AS manager,
       length(path) AS livelli
ORDER BY livelli
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["impiegato"], "->", r["manager"], "(", r["livelli"], "livelli)")


In [ ]:
# --- ShortestPath: cammino minimo tra due impiegati ---
# nodes(path) estrae i nodi, [n IN ... | n.name] li trasforma in nomi
query = """
MATCH (a:Person {employee_id: 1}), (b:Person {employee_id: 100})
MATCH path = shortestPath((a)-[:WORKED_ON*]-(b))
RETURN [n IN nodes(path) | n.name] AS percorso,
       length(path) AS hop
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print("Percorso (" + str(r["hop"]) + " hop):", " -> ".join(r["percorso"]))


## 4. MERGE vs CREATE: Idempotenza


- **CREATE**: inserisce sempre. Se il nodo esiste (UNIQUE constraint), fallisce.
- **MERGE**: cerca il pattern. Se non esiste, lo crea.
- Idempotente: eseguire N volte produce lo stesso risultato.
- `ON MATCH SET` / `ON CREATE SET` per comportamenti distinti.


In [ ]:
# --- MERGE idempotente ---
# La prima volta crea, la seconda volta trova e basta
query = """
MERGE (p:Person {employee_id: 999})
  SET p.name = "Test User", p.location = "Test"
RETURN p.employee_id, p.name, p.location
"""
records, _, _ = driver.execute_query(query)
print("1a esecuzione:", records[0]["name"])
records, _, _ = driver.execute_query(query)
print("2a esecuzione (idempotente):", records[0]["name"])

# Pulizia
driver.execute_query('MATCH (p:Person {employee_id: 999}) DETACH DELETE p')
print("Pulito")


In [ ]:
# --- MERGE su relazione con ON CREATE/MATCH SET ---
query = """
MATCH (p:Person {employee_id: 1}), (s:Skill {name: 'Python'})
MERGE (p)-[h:HAS_SKILL]->(s)
  ON CREATE SET h.proficiency = 'Senior'
  ON MATCH SET h.proficiency = 'Senior'
RETURN p.name, s.name, h.proficiency
"""
records, _, _ = driver.execute_query(query)
print(records[0]["p.name"], "-", records[0]["s.name"], "-", records[0]["h.proficiency"])


## 5. Filtri WHERE Avanzati


AND, OR, NOT, IN [...], CONTAINS, STARTS WITH, ENDS WITH, IS NULL.
Possono essere usati sia nel WHERE che direttamente nel MATCH.


In [ ]:
# --- AND/OR con IN e STARTS WITH ---
query = """
MATCH (p:Person)
WHERE (p.location IN ["Roma", "Milano"])
  AND p.name STARTS WITH "M"
RETURN p.name, p.location, p.title
ORDER BY p.name
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["p.name"], "|", r["p.location"], "|", r["p.title"])


In [ ]:
# --- CONTAINS: ricerca testuale su skill ---
query = """
MATCH (s:Skill)
WHERE s.name CONTAINS "cloud"
RETURN s.name AS skill
ORDER BY skill
"""
records, _, _ = driver.execute_query(query)
print("Skill con \"cloud\":", len(records))
for r in records:
    print(" -", r["skill"])


## 6. Aggregazioni


count(*), collect(), avg(), sum(), min(), max().
Le colonne non aggregate sono automaticamente grouping key.


In [ ]:
# --- Skill più diffuse ---
query = """
MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill, count(*) AS persone
ORDER BY persone DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "|", r["persone"])


In [ ]:
# --- Proficiency media per skill ---
# h.proficiency è la proprietà sulla relazione HAS_SKILL
query = """
MATCH (p:Person)-[h:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill,
       round(avg(h.proficiency), 1) AS media,
       count(*) AS persone
ORDER BY media DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "| media:", r["media"], "|", r["persone"], "persone")


In [ ]:
# --- COLLECT: lista progetti per persona ---
query = """
MATCH (p:Person)-[:WORKED_ON]->(proj:Project)
RETURN p.name AS persona,
       collect(proj.name) AS progetti,
       count(proj) AS num
ORDER BY num DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    proj_list = ", ".join(r["progetti"])
    print(r["persona"], "|", r["num"], "progetti:", proj_list)


## 7. Ordinamento e Paginazione


ORDER BY ASC/DESC (anche multipli), LIMIT n, SKIP n per paginazione.


In [ ]:
# --- Paginazione: seconda pagina (record 11-20) ---
query = """
MATCH (p:Person)
RETURN p.name AS nome, p.location AS location
ORDER BY nome
SKIP 10 LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "|", r["location"])


## 8. Subquery con CALL { ... }


CALL { subquery } permette di annidare query. UNION combina risultati.
Le subquery correlate possono riferire variabili esterne.


In [ ]:
# --- UNION: skill e progetti in una lista ---
query = """
CALL { MATCH (s:Skill) RETURN s.name AS nome, 'Skill' AS tipo }
UNION
CALL { MATCH (p:Project) RETURN p.name AS nome, 'Progetto' AS tipo }
RETURN nome, tipo
ORDER BY nome
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "|", r["tipo"])


In [ ]:
# --- Subquery correlate: conteggi multipli per persona ---
# Ogni CALL { } si riferisce alla persona corrente (p)
query = """
MATCH (p:Person)
RETURN p.name AS nome,
       CALL { MATCH (p)-[:HAS_SKILL]->(s:Skill) RETURN count(*) } AS num_skill,
       CALL { MATCH (p)-[:WORKED_ON]->(proj:Project) RETURN count(*) } AS num_progetti
ORDER BY num_skill DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "| skill:", r["num_skill"], "| progetti:", r["num_progetti"])


## 9. Date, Liste e Mappe


In [ ]:
# date() = data odierna, date('2026-07-15') = data specifica
query = "RETURN date() AS oggi, date('2026-07-15') AS prima_sessione"
records, _, _ = driver.execute_query(query)
print("Oggi:", records[0]["oggi"])
print("Prima sessione:", records[0]["prima_sessione"])


In [ ]:
# range(start, end) + UNWIND = genera righe da una lista
query = """
UNWIND range(2020, 2026) AS anno
RETURN anno
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r['anno'])


In [ ]:
# List comprehension: [x IN list WHERE cond | expr]
# Filtra numeri > 2 e moltiplica per 10
query = """
WITH [1, 2, 3, 4, 5] AS numeri
RETURN [x IN numeri WHERE x > 2 | x * 10] AS trasformati
"""
records, _, _ = driver.execute_query(query)
print("Trasformati:", records[0]["trasformati"])


## 10. Query Tuning: PROFILE ed EXPLAIN


EXPLAIN = piano stimato (non esegue).
PROFILE = esegue e mostra dbHits, rows.
NodeIndexSeek = buono (usa indice). AllNodesScan = cattivo (scan totale).


In [ ]:
# EXPLAIN: piano stimato senza esecuzione
query = "EXPLAIN MATCH (p:Person)-[:HAS_SKILL]->(s:Skill) WHERE p.location = 'Roma' RETURN p.name, s.name"
records, summary, _ = driver.execute_query(query)
print("=== PIANO STIMATO ===")
for op in summary.plan.operators:
    print(' ', op)


In [ ]:
# PROFILE: esecuzione con metriche reali
query = "PROFILE MATCH (p:Person {location: 'Roma'})-[:HAS_SKILL]->(s:Skill) RETURN p.name, s.name"
records, summary, _ = driver.execute_query(query)
print("=== RISULTATI ===")
for r in records:
    print(r["p.name"], "|", r["s.name"])


## 11. APOC: Procedure Standard


APOC = libreria ufficiale di procedure. Su AuraDB molte sono già disponibili.
apoc.coll.*, apoc.text.*, apoc.load.csv, apoc.periodic.iterate.


In [ ]:
# Verifica disponibilita APOC
try:
    records, _, _ = driver.execute_query('RETURN apoc.version() AS versione')
    print("APOC versione:", records[0]["versione"])
except Exception as e:
    print("APOC non disponibile:", e)


In [ ]:
# apoc.coll.zip = unisce due liste come Python zip()
query = "RETURN apoc.coll.zip([1, 2, 3], ['a', 'b', 'c']) AS zippato"
records, _, _ = driver.execute_query(query)
print("Zippato:", records[0]["zippato"])


In [ ]:
# apoc.text.clean = normalizza testo (rimuove accenti, speciali)
query = "RETURN apoc.text.clean('Cypher Query Language!') AS pulito"
records, _, _ = driver.execute_query(query)
print("Pulito:", records[0]["pulito"])


## 12. Esercizi


Scrivete query Cypher per questi problemi sul dataset Org Graph.
Potete usare driver.execute_query() qui o Neo4j Browser.

**Proprietà del modello:**
- Person: employee_id, name, location, title, department
- Skill: name
- Project: project_id, name
- HAS_SKILL: proficiency
- WORKED_ON: role


### Esercizio 1 — Skill più diffuse


Trovate le 5 skill più diffuse tra i dipendenti.


In [ ]:
query = """
MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill, count(*) AS persone
ORDER BY persone DESC
LIMIT 5
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["skill"], "-", r["persone"], "persone")


### Esercizio 2 — Progetti per persona


Per ogni persona, contate i progetti. Mostrate le prime 10.


In [ ]:
query = """
MATCH (p:Person)-[:WORKED_ON]->(proj:Project)
RETURN p.name AS persona, count(proj) AS progetti
ORDER BY progetti DESC
LIMIT 10
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["persona"], "-", r["progetti"], "progetti")


### Esercizio 3 — Cammino minimo


shortestPath tra due impiegati a scelta via progetti condivisi.


In [ ]:
query = """
MATCH (a:Person {employee_id: 1}), (b:Person {employee_id: 50})
MATCH path = shortestPath((a)-[:WORKED_ON*]-(b))
RETURN [n IN nodes(path) | n.name] AS percorso,
       length(path) AS hop
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(" -> ".join(r["percorso"]), "(", r["hop"], "hop)")


### Esercizio 4 — Skill overlap


Coppie di impiegati con almeno 3 skill in comune.


In [ ]:
query = """
MATCH (p1:Person)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(p2:Person)
WHERE id(p1) < id(p2)
WITH p1.name AS p1n, p2.name AS p2n, collect(s.name) AS comuni
WHERE size(comuni) >= 3
RETURN p1n, p2n, comuni
ORDER BY p1n
LIMIT 20
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["p1n"], "|", r["p2n"], "|", ", ".join(r["comuni"]))


### Esercizio 5 — Subquery correlate


Nome, skill e progetti per ogni persona via CALL { }.


In [ ]:
query = """
MATCH (p:Person)
RETURN p.name AS nome,
       CALL { MATCH (p)-[:HAS_SKILL]->(s:Skill) RETURN count(*) } AS num_skill,
       CALL { MATCH (p)-[:WORKED_ON]->(proj:Project) RETURN count(*) } AS num_progetti
ORDER BY nome
LIMIT 15
"""
records, _, _ = driver.execute_query(query)
for r in records:
    print(r["nome"], "| skill:", r["num_skill"], "| progetti:", r["num_progetti"])


### Esercizio 6 — PROFILE


Analizzate una query con PROFILE. Cercate operatori inefficienti.


In [ ]:
query = """PROFILE
MATCH (p1:Person)-[:HAS_SKILL]->(s:Skill)<-[:HAS_SKILL]-(p2:Person)
WHERE id(p1) < id(p2)
RETURN p1.name, p2.name, count(s) AS skill_comuni
ORDER BY skill_comuni DESC
LIMIT 10
"""
records, summary, _ = driver.execute_query(query)
print("=== OPERATORI ===")
for op in summary.plan.operators:
    print(' ', op)
print("\n=== RISULTATI ===")
for r in records:
    print(r["p1.name"], "|", r["p2.name"], "|", r["skill_comuni"], "skill")


## Riepilogo


| Concetto | Esempio |
|----------|--------|
| Path variabile | (p)-[:REPORTS_TO*1..4]->(m) |
| ShortestPath | shortestPath((a)-[:WORKED_ON*]-(b)) |
| MERGE idempotente | MERGE (p:Person {id: n}) SET ... |
| Filtri WHERE | location IN [...] AND name CONTAINS ... |
| Aggregazioni | count(*), collect(), avg() |
| Subquery | CALL { MATCH ... } |
| PROFILE/EXPLAIN | diagnostica performance |
| APOC | apoc.coll.*, apoc.text.* |


In [ ]:
driver.close()
print("Connessione chiusa")
